# TRM exploratory analysis

TRM is the USD/COP representative market exchange rate, expressed as Colombian pesos per US dollar. The analysis uses consecutive observed values. A non-publication day is not automatically a missing observation.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from trm_forecast.data.audit import audit_trm

DATA_PATH = next(path / "data/interim/trm_normalized.csv" for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/interim/trm_normalized.csv").exists())
data = pd.read_csv(DATA_PATH)
report = audit_trm(data)
data.info()
display(data.head())
display(pd.DataFrame([{"shape": data.shape, "earliest": report.earliest_date, "latest": report.latest_date, "valid_observations": report.observation_count}]))

In [ ]:
analysis = data.copy()
analysis["date"] = pd.to_datetime(analysis["date"], errors="coerce")
analysis["trm"] = pd.to_numeric(analysis["trm"], errors="coerce")
analysis = analysis.dropna(subset=["date", "trm"]).sort_values("date").copy()
analysis["daily_change_cop"] = analysis["trm"].diff()
analysis["log_return"] = np.log(analysis["trm"]).diff()
analysis["pct_change"] = analysis["trm"].pct_change() * 100

display(analysis["trm"].describe().to_frame("TRM (COP per USD)"))
display(analysis[["date", "trm", "daily_change_cop", "pct_change", "log_return"]])

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 11), sharex=True)
axes[0].plot(analysis["date"], analysis["trm"], marker="o")
axes[0].set_ylabel("COP per USD")
axes[0].set_title("Observed TRM level")
axes[1].bar(analysis["date"], analysis["daily_change_cop"], width=0.8)
axes[1].set_ylabel("COP")
axes[1].set_title("Change from previous observed value")
axes[2].bar(analysis["date"], analysis["log_return"], width=0.8)
axes[2].set_ylabel("log return")
axes[2].set_title("Log return: ln(TRM_t / TRM_(t-1))")
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(analysis["trm"], bins=min(10, max(1, len(analysis))), edgecolor="black")
axes[0].set_xlabel("COP per USD")
axes[0].set_title("TRM level distribution")
axes[1].hist(analysis["daily_change_cop"].dropna(), bins=min(10, max(1, len(analysis)-1)), edgecolor="black")
axes[1].set_xlabel("COP change")
axes[1].set_title("Observed changes distribution")
fig.tight_layout()

In [ ]:
changes = analysis.dropna(subset=["daily_change_cop"]).copy()
if len(changes) >= 3:
    threshold = changes["daily_change_cop"].abs().quantile(0.95)
    unusual = changes[changes["daily_change_cop"].abs() >= threshold]
else:
    threshold = np.nan
    unusual = changes.iloc[0:0]
display(unusual[["date", "trm", "daily_change_cop", "pct_change"]])
print("These are screening candidates for source/context review, not automatic errors.")

In [ ]:
audit_table = pd.DataFrame([{key: value for key, value in report.to_dict().items() if not isinstance(value, (tuple, list))}]).T.rename(columns={0: "value"})
display(audit_table)
display(pd.DataFrame(report.candidate_missing_periods))

## Interpretation and limitations

The level is a price in COP per USD; changes and returns measure movement between consecutive published observations and therefore answer different questions. Visual fit to historical levels is not enough for forecasting: errors, stability across origins, forecast horizons, and leakage-aware evaluation matter. A historical pattern does not establish persistence over the next 1–10 business days.

This file is only five observations long, so distributional and unusually-large-change results are descriptive and not statistically stable. The local file has not been independently checked against the live source in this notebook; any suspicious discontinuity requires source and institutional-calendar review.